# IOTN-AC (Front View) — Prediktor Multi-Task-Ready + Interpretasi

Fokus: **skor IOTN-AC 1–10 dari view frontal saja** (OMNI menyediakan 5 view, tapi tahap
ini hanya frontal). Arsitekturnya **disiapkan multi-task** sesuai rancangan pipeline:

- **Kepala AC** — aktif & dilatih sekarang (label AC 1–10 tersedia).
- **Kepala kasus MOCDO** (9 komponen) — **sudah didefinisikan tapi belum dilatih**, karena
  annotator belum memberi label komponennya. Tinggal diaktifkan (`AKTIFKAN_KASUS = True`)
  begitu label + 5 view tersedia → lalu diagregasi menjadi IOTN-DHC.
- **Grad-CAM** — interpretasi "model melihat ke mana" **tanpa perlu label kasus**, bisa
  dipakai sekarang.

Outline: 1) Setup · 2) Muat label AC · 3) Dataset · 4) Model (AC aktif + MOCDO stub) ·
5) Latih AC · 6) Evaluasi · 7) Grad-CAM · 8) Mengaktifkan kepala MOCDO (nanti).

## 1. Setup — path, perangkat, parameter

In [ ]:
import os, time, collections
import numpy as np, openpyxl
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import torchvision.transforms.functional as TF
from PIL import Image
import matplotlib.pyplot as plt

try:    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError: _HERE = os.getcwd()
PROJECT_ROOT = os.path.dirname(_HERE) if os.path.basename(_HERE) == 'notebooks' else _HERE
DATA_RAW   = os.path.join(PROJECT_ROOT, 'data', 'raw')
LABEL_XLSX = os.path.join(PROJECT_ROOT, 'data', 'labels', 'new ac report.xlsx')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models'); os.makedirs(MODELS_DIR, exist_ok=True)
STAMP   = time.strftime('%Y-%m-%d_%H%M%S')
RUN_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'runs', STAMP + '_ac09')
os.makedirs(RUN_DIR, exist_ok=True)

PERANGKAT = ('cuda' if torch.cuda.is_available()
             else ('mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available()
                   else 'cpu'))
BACKBONE = 'mobilenet_v3_small'
FOLDER = {'Train': 'train', 'Val': 'val', 'Test': 'test'}   # view frontal (OMNI front)
# 9 komponen MOCDO (kepala distub sampai ada label)
KASUS = ['missing', 'overjet', 'crossbite', 'displacement', 'overbite',
         'spacing', 'rotation', 'canine_ectopic', 'midline_deviation']
AKTIFKAN_KASUS = False    # jadikan True saat label 9 komponen sudah tersedia

assert os.path.exists(LABEL_XLSX), f'Label tidak ditemukan: {LABEL_XLSX}'
print('Perangkat:', PERANGKAT, '| backbone:', BACKBONE)
print('Kepala kasus MOCDO:', 'AKTIF' if AKTIFKAN_KASUS else 'distub (belum ada label)')

## 2. Muat label AC (view frontal)

Membaca `new ac report.xlsx` (sheet Train/Val/Test), kolom **Nama File** & **Grade AC**, dan memasangkannya dengan foto di `data/raw/{train,val,test}`.

In [ ]:
def grade_angka(v):
    if isinstance(v, bool) or v is None: return None
    if isinstance(v, (int, float)) and float(v).is_integer():
        g = int(v); return g if 1 <= g <= 10 else None
    return None

def muat_label():
    wb = openpyxl.load_workbook(LABEL_XLSX)
    data = {'train': [], 'val': [], 'test': []}
    for sheet, sub in FOLDER.items():
        folder = os.path.join(DATA_RAW, sub)
        for baris in wb[sheet].iter_rows(min_row=2, values_only=True):
            nama, grade = baris[1], baris[3]
            g = grade_angka(grade)
            if nama and g is not None:
                p = os.path.join(folder, str(nama).strip())
                if os.path.exists(p):
                    data[sub].append((p, g))
    return data

DATA = muat_label()
for s in ['train', 'val', 'test']:
    dist = collections.Counter(g for _, g in DATA[s])
    print(f'{s:5s}: {len(DATA[s]):4d} foto | sebaran grade {dict(sorted(dist.items()))}')

## 2b. ROI area gigi (heuristik atau deteksi Roboflow, dengan cache)

Foto dipangkas ke area gigi supaya model fokus ke gigi (bukan retraktor/bibir/latar) dan
Grad-CAM jatuh di gigi. Dua sumber ROI:

- `SUMBER_ROI = 'heuristik'` — aturan warna + region-growing (gratis, offline, berplafon).
- `SUMBER_ROI = 'roboflow'` — model **deteksi gigi** `front-intraoral-tooth-numbering` (modalitas
  sama dengan OMNI); kotak semua gigi digabung jadi ROI. Perlu `ROBOFLOW_API_KEY`.

ROI dihitung **sekali** untuk semua foto lalu **di-cache ke CSV** (`outputs/roi_box_*.csv`) —
jadi tidak ada panggilan API berulang saat training. Bila deteksi gagal, otomatis jatuh ke
heuristik. Hapus berkas cache untuk menghitung ulang.

In [ ]:
# ============ ROI area gigi: heuristik ATAU deteksi Roboflow (dengan cache) ============
import csv as _csv
SUMBER_ROI       = 'heuristik'    # 'heuristik' | 'roboflow'
ROBOFLOW_API_KEY = ''             # untuk 'roboflow': app.roboflow.com/settings/api
MODEL_ID         = 'front-intraoral-tooth-numbering/1'
PAKAI_ROI        = True
ROI_CACHE = os.path.join(PROJECT_ROOT, 'outputs', f'roi_box_{SUMBER_ROI}.csv')

# --- sumber 1: heuristik warna + region-growing (sama seperti notebook 8) ---
def _dil(m):
    o = m.copy(); o[:-1] |= m[1:]; o[1:] |= m[:-1]; o[:, :-1] |= m[:, 1:]; o[:, 1:] |= m[:, :-1]; return o
def _ero(m): return ~_dil(~m)
def _op(m, k):
    for _ in range(k): m = _ero(m)
    for _ in range(k): m = _dil(m)
    return m
def _cl(m, k):
    for _ in range(k): m = _dil(m)
    for _ in range(k): m = _ero(m)
    return m
def roi_box_heur(img, W=220, pad=0.05):
    w0, h0 = img.size
    sm = np.asarray(img.resize((W, int(W * h0 / w0))), float)
    R, G, B = sm[..., 0], sm[..., 1], sm[..., 2]
    br = (R + G + B) / 3; rg = R - G
    mx, mn = sm.max(2), sm.min(2); sat = (mx - mn) / (mx + 1e-6)
    mask = (br > 145) & (rg < 50) & (sat < 0.40)
    mask = _op(mask, 1); mask = _cl(mask, 4)
    h, w = mask.shape
    if mask.sum() < 40:
        return (int(w0*.2), int(h0*.2), int(w0*.8), int(h0*.8))
    seed = np.zeros_like(mask); seed[int(h*.28):int(h*.74), int(w*.15):int(w*.85)] = True; seed &= mask
    if seed.sum() == 0: seed = mask.copy()
    for _ in range(W):
        nx = _dil(seed) & mask
        if nx.sum() == seed.sum(): break
        seed = nx
    ys, xs = np.where(seed); sx, sy = w0 / w, h0 / h
    x0, x1, y0, y1 = xs.min()*sx, xs.max()*sx, ys.min()*sy, ys.max()*sy
    dx, dy = (x1 - x0) * pad, (y1 - y0) * pad
    return (int(max(x0-dx,0)), int(max(y0-dy,0)), int(min(x1+dx,w0)), int(min(y1+dy,h0)))

# --- sumber 2: deteksi gigi Roboflow via HTTP (union kotak = ROI) ---
def roi_box_roboflow(img, pad=0.05):
    import io, json as _json, base64, urllib.request
    w0, h0 = img.size
    buf = io.BytesIO(); img.convert('RGB').save(buf, format='JPEG')
    url = f'https://detect.roboflow.com/{MODEL_ID}?api_key={ROBOFLOW_API_KEY}'
    req = urllib.request.Request(url, data=base64.b64encode(buf.getvalue()),
                                 headers={'Content-Type': 'application/x-www-form-urlencoded'})
    with urllib.request.urlopen(req, timeout=30) as r:
        pred = _json.loads(r.read()).get('predictions', [])
    if not pred: raise ValueError('tak ada gigi terdeteksi')
    x0 = min(p['x'] - p['width']/2  for p in pred); y0 = min(p['y'] - p['height']/2 for p in pred)
    x1 = max(p['x'] + p['width']/2  for p in pred); y1 = max(p['y'] + p['height']/2 for p in pred)
    dx, dy = (x1 - x0) * pad, (y1 - y0) * pad
    return (int(max(x0-dx,0)), int(max(y0-dy,0)), int(min(x1+dx,w0)), int(min(y1+dy,h0)))

# --- hitung ROI semua foto SEKALI, cache ke CSV ---
SEMUA = [p for s in ['train', 'val', 'test'] for p, _ in DATA[s]]
ROI_BOX = {}
if os.path.exists(ROI_CACHE):
    for row in _csv.reader(open(ROI_CACHE)):
        if row and row[0] != 'path': ROI_BOX[row[0]] = tuple(int(v) for v in row[1:5])
    print(f'ROI dimuat dari cache ({len(ROI_BOX)} kotak): {os.path.relpath(ROI_CACHE, PROJECT_ROOT)}')
else:
    print(f'Menghitung ROI ({SUMBER_ROI}) untuk {len(SEMUA)} foto (sekali jalan)...')
    n_fb = 0
    for p in SEMUA:
        im = Image.open(p).convert('RGB')
        try:
            box = roi_box_roboflow(im) if (SUMBER_ROI == 'roboflow' and ROBOFLOW_API_KEY) else roi_box_heur(im)
        except Exception:
            box = roi_box_heur(im); n_fb += 1
        ROI_BOX[p] = box
    os.makedirs(os.path.dirname(ROI_CACHE), exist_ok=True)
    with open(ROI_CACHE, 'w', newline='') as fh:
        w = _csv.writer(fh); w.writerow(['path', 'x0', 'y0', 'x1', 'y1'])
        for p, b in ROI_BOX.items(): w.writerow([p, *b])
    print(f'ROI disimpan ke {os.path.relpath(ROI_CACHE, PROJECT_ROOT)} (fallback heuristik: {n_fb})')

def crop_roi(path, img):
    return img.crop(ROI_BOX[path]) if (PAKAI_ROI and path in ROI_BOX) else img

# pratinjau 6 foto
_prev = DATA['train'][:6]
fig, ax = plt.subplots(1, 6, figsize=(16, 3))
for a, (p, g) in zip(ax, _prev):
    a.imshow(crop_roi(p, Image.open(p).convert('RGB'))); a.axis('off'); a.set_title(f'grade {g}', fontsize=9)
fig.suptitle(f'Pratinjau ROI ({SUMBER_ROI}) — masukan model', fontsize=11)
plt.tight_layout(); plt.show()
print('ROI:', SUMBER_ROI, '| PAKAI_ROI:', PAKAI_ROI)

## 3. Dataset & DataLoader (frontal)

In [ ]:
RATA, SIMPANG = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
tf_train = transforms.Compose([
    transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.15, 0.15, 0.15),
    transforms.ToTensor(), transforms.Normalize(RATA, SIMPANG)])
tf_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(), transforms.Normalize(RATA, SIMPANG)])

class DataAC(Dataset):
    def __init__(self, pasangan, tf): self.pasangan = pasangan; self.tf = tf
    def __len__(self): return len(self.pasangan)
    def __getitem__(self, i):
        p, g = self.pasangan[i]
        img = crop_roi(p, Image.open(p).convert('RGB'))
        return self.tf(img), float(g)

def loader(split, tf, shuffle):
    return DataLoader(DataAC(DATA[split], tf), batch_size=16, shuffle=shuffle, num_workers=0)

dl_train = loader('train', tf_train, True)
dl_val   = loader('val',   tf_eval,  False)
dl_test  = loader('test',  tf_eval,  False)
print('batch train/val/test:', len(dl_train), len(dl_val), len(dl_test))

## 4. Model — backbone + kepala AC (aktif) + kepala MOCDO (stub)

Satu backbone berbagi. **Kepala AC** memetakan fitur ke skor 1–10 (sigmoid-scaling).
**Kepala kasus** = 9 head kecil, sudah ada di model tetapi hanya ikut dilatih bila
`AKTIFKAN_KASUS = True` dan label tersedia.

In [ ]:
class MultiTaskAC(nn.Module):
    def __init__(self, nama=BACKBONE, dropout=0.3, kasus=KASUS):
        super().__init__()
        bb = getattr(models, nama)(weights='DEFAULT')
        self.features = bb.features            # backbone konvolusi
        self.pool = bb.avgpool
        in_feat = bb.classifier[0].in_features
        self.kepala_ac = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(in_feat, 256), nn.BatchNorm1d(256),
            nn.ReLU(inplace=True), nn.Dropout(dropout), nn.Linear(256, 1))
        # 9 kepala kasus MOCDO (multi-label) — distub sampai ada label
        self.kepala_kasus = nn.ModuleDict({k: nn.Linear(in_feat, 1) for k in kasus})
        self.lo, self.rentang = 1.0, 9.0       # skala keluaran AC -> [1,10]

    def forward(self, x, dengan_kasus=False):
        f = torch.flatten(self.pool(self.features(x)), 1)
        keluar = {'ac': self.lo + self.rentang * torch.sigmoid(self.kepala_ac(f))}
        if dengan_kasus:
            keluar['kasus'] = {k: h(f) for k, h in self.kepala_kasus.items()}
        return keluar

model = MultiTaskAC().to(PERANGKAT)
n_ac = sum(p.numel() for p in model.kepala_ac.parameters())
n_ks = sum(p.numel() for p in model.kepala_kasus.parameters())
print(f'total parameter : {sum(p.numel() for p in model.parameters()):,}')
print(f'kepala AC       : {n_ac:,} parameter (dilatih)')
print(f'kepala MOCDO    : {n_ks:,} parameter (9 head, {"dilatih" if AKTIFKAN_KASUS else "distub / beku"})')

## 5. Latih (AC saja)

Loss = MSE pada skor Aced. Bila nanti `AKTIFKAN_KASUS = True` dan Dataset mengembalikan
label kasus, tinggal tambahkan BCE per komponen ke total loss (kerangkanya sudah ada di
komentar). Bobot backbone awal dibekukan sebentar lalu dibuka (fine-tune ringan).

In [ ]:
def acak(s=42):
    import random; random.seed(s); np.random.seed(s); torch.manual_seed(s)

@torch.no_grad()
def prediksi(dl):
    model.eval(); ps, ys = [], []
    for x, y in dl:
        ps.append(model(x.to(PERANGKAT))['ac'].squeeze(1).cpu().numpy()); ys.append(y.numpy())
    return np.concatenate(ps), np.concatenate(ys)

def latih(epoch=15, lr=1e-3, beku_awal=4):
    acak()
    # bekukan sebagian blok awal backbone
    for i, blok in enumerate(model.features):
        for p in blok.parameters(): p.requires_grad = (i >= beku_awal)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                            lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epoch)
    mse = nn.MSELoss()
    terbaik, bobot = 1e9, None
    for ep in range(epoch):
        model.train()
        for x, y in dl_train:
            x, y = x.to(PERANGKAT), y.float().to(PERANGKAT)   # cast float32 dulu (MPS tak dukung float64)
            out = model(x, dengan_kasus=AKTIFKAN_KASUS)
            loss = mse(out['ac'].squeeze(1), y)
            # --- KERANGKA multi-task (aktif saat label kasus tersedia) ---
            # if AKTIFKAN_KASUS:
            #     bce = nn.BCEWithLogitsLoss()
            #     for k in KASUS:
            #         loss = loss + 0.3 * bce(out['kasus'][k].squeeze(1), label_kasus[k])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        sched.step()
        pv, yv = prediksi(dl_val); mae = float(np.abs(np.clip(np.round(pv),1,10) - yv).mean())
        tanda = ''
        if mae < terbaik:
            terbaik = mae; bobot = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}; tanda = '  <- terbaik'
        print(f'  epoch {ep+1:2d}/{epoch}  val MAE {mae:.3f}{tanda}')
    model.load_state_dict(bobot)
    return terbaik

print('Melatih kepala AC (view frontal)...\n')
val_mae = latih()
torch.save(model.state_dict(), os.path.join(MODELS_DIR, 'ac_multitask_frontview.pt'))
print(f'\nMAE validasi terbaik: {val_mae:.3f} | bobot disimpan ke models/ac_multitask_frontview.pt')

## 6. Evaluasi AC pada data uji

In [ ]:
def qwk(b, p, K=10):
    b = np.asarray(b, int) - 1; p = np.clip(np.asarray(p, int) - 1, 0, K - 1)
    O = np.zeros((K, K));
    for i, j in zip(b, p): O[i, j] += 1
    w = (np.arange(K)[:, None] - np.arange(K)[None, :]) ** 2 / (K - 1) ** 2
    E = np.outer(np.bincount(b, minlength=K), np.bincount(p, minlength=K)) / len(b)
    return 1 - (w * O).sum() / max((w * E).sum(), 1e-9)

pt, yt = prediksi(dl_test)
pr = np.clip(np.round(pt), 1, 10)
print(f'MAE   : {np.abs(pr - yt).mean():.3f}')
print(f'RMSE  : {np.sqrt(((pr - yt) ** 2).mean()):.3f}')
print(f'tepat : {(pr == yt).mean():.1%}')
print(f'±1    : {(np.abs(pr - yt) <= 1).mean():.1%}')
print(f'QWK   : {qwk(yt, pr):.3f}')

## 7. Interpretasi — Grad-CAM

Menyorot bagian foto yang paling memengaruhi prediksi AC, **tanpa perlu label kasus**.
Berguna untuk memeriksa apakah model benar-benar melihat gigi (bukan latar/retraktor).

In [ ]:
class GradCAM:
    def __init__(self, model, lapisan):
        self.model = model; self.act = None; self.grad = None
        lapisan.register_forward_hook(lambda m, i, o: setattr(self, 'act', o.detach()))
        lapisan.register_full_backward_hook(lambda m, gi, go: setattr(self, 'grad', go[0].detach()))
    def __call__(self, x):
        self.model.eval()
        self.model.zero_grad()
        skor = self.model(x)['ac']
        skor.sum().backward()
        bobot = self.grad.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((bobot * self.act).sum(1))
        cam = cam / (cam.amax(dim=(1, 2), keepdim=True) + 1e-8)
        return cam.cpu().numpy(), skor.detach().cpu().numpy().ravel()

cam = GradCAM(model, model.features[-1])
contoh = DATA['test'][:6]
fig, axes = plt.subplots(2, 6, figsize=(17, 6))
for k, (p, g) in enumerate(contoh):
    img = crop_roi(p, Image.open(p).convert('RGB')).resize((224, 224))
    x = tf_eval(img).unsqueeze(0).to(PERANGKAT)
    heat, skor = cam(x)
    h = np.asarray(Image.fromarray((heat[0] * 255).astype('uint8')).resize((224, 224))) / 255
    axes[0, k].imshow(img); axes[0, k].axis('off')
    axes[0, k].set_title(f'AC~{skor[0]:.1f} (label {g})', fontsize=9)
    axes[1, k].imshow(img); axes[1, k].imshow(h, cmap='jet', alpha=0.45); axes[1, k].axis('off')
    axes[1, k].set_title('Grad-CAM', fontsize=9)
fig.suptitle('Interpretasi: area yang memengaruhi prediksi AC', fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(RUN_DIR, 'gradcam_ac.png'), dpi=130, bbox_inches='tight')
plt.show()
print('Grad-CAM disimpan ke', os.path.relpath(RUN_DIR, PROJECT_ROOT) + '/gradcam_ac.png')

## 8. (Nanti) Mengaktifkan kepala MOCDO → IOTN-DHC

Saat annotator sudah memberi **label 9 komponen MOCDO** (dan idealnya **5 view** tersedia):

1. Ubah `Dataset` agar mengembalikan juga vektor label kasus per foto.
2. Set `AKTIFKAN_KASUS = True` — kerangka loss multi-task di bagian 5 tinggal di-uncomment
   (MSE untuk AC + BCE untuk tiap komponen).
3. Untuk 5 view: bungkus `MultiTaskAC` agar memproses 5 citra lalu **fuse** (concat + dense)
   sebelum kepala — sesuai diagram pipeline (`outputs/diagrams/`).
4. **Agregasi MOCDO → DHC 1–5** memakai aturan IOTN (grade komponen terparah menang);
   ini logika sesudah model, bukan lapisan yang dilatih.

Selama tahap ini, kepala MOCDO tetap ada di model (bobotnya acak & beku) sehingga struktur
tak berubah — hanya perlu label untuk menyalakannya.